# Training monitor

Read a run's append-only structured metrics and bounded log tail without accessing the training process.

In [ ]:
import os
import sys
from pathlib import Path


def required_path(name):
    value = os.environ.get(name)
    assert value, f"Missing required environment variable: {name}"
    return Path(value).expanduser().resolve()


repo_root = required_path("DFDHR_REPO_ROOT")
runtime_root = required_path("DFDHR_RUNTIME_ROOT")
metrics_path = required_path("DFDHR_METRICS_JSONL")
assert metrics_path.is_file()
assert metrics_path.is_relative_to(runtime_root)
sys.path.insert(0, str(repo_root / "training"))
from structured_metrics import read_complete_jsonl


In [ ]:
events = read_complete_jsonl(metrics_path)
assert events, "No complete metric events found"
assert all(event.get("schema_version") == 1 for event in events)
train_events = [event for event in events if event.get("event") == "train_batch"]
validation_events = [event for event in events if event.get("phase") in {"val", "test"}]
print("Complete events:", len(events))
print("Train events:", len(train_events))
print("Validation/evaluation events:", len(validation_events))
print("Latest global step:", events[-1].get("global_step"))
print("Metrics role: DFDHR_RUNTIME_ROOT/<run>/metrics.jsonl")


In [ ]:
assert train_events, "At least one train_batch event is required for monitoring"
columns = ("global_step", "loss", "learning_rate", "step_time_seconds", "data_time_seconds", "cuda_memory_allocated_bytes", "disk_free_bytes")
print(" | ".join(columns))
print("-" * 112)
for event in train_events[-20:]:
    print(" | ".join(str(event.get(column)) for column in columns))
loss_change = train_events[-1].get("loss") - train_events[0].get("loss")
print(f"Loss change across recorded window: {loss_change:+.6f}")
print("Latest validation metrics:", validation_events[-1].get("metrics") if validation_events else None)


In [ ]:
log_path = Path(os.environ.get("DFDHR_TRAINING_LOG", metrics_path.parent / "training.log")).expanduser().resolve()
assert log_path.is_relative_to(runtime_root)
log_lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines() if log_path.is_file() else []
print("Training log tail (up to 40 lines):")
print("\n".join(log_lines[-40:]))
